In [ ]:

import csv

# Create sample CSV training data
data = [
    ["Sky", "AirTemp", "Humidity", "Wind", "Water", "Forecast", "EnjoySport"],
    ["Sunny", "Warm", "Normal", "Strong", "Warm", "Same", "Yes"],
    ["Sunny", "Warm", "High", "Strong", "Warm", "Same", "Yes"],
    ["Rainy", "Cold", "High", "Strong", "Warm", "Change", "No"],
    ["Sunny", "Warm", "High", "Strong", "Cool", "Change", "Yes"]
]

with open("training_data.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(data)

# Read CSV file
with open("training_data.csv", "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

attributes = rows[0][:-1]
examples = rows[1:]
domains = [
    list(dict.fromkeys(row[i] for row in examples))
    for i in range(len(attributes))
]

def covers(h, x):
    return all(a == "?" or a == b for a, b in zip(h, x))

def more_general(h1, h2):
    return all(a == "?" or a == b for a, b in zip(h1, h2))

def generalize(s, x):
    return [a if a == b else "?" for a, b in zip(s, x)]

def remove_duplicates(hypotheses):
    result = []
    for h in hypotheses:
        if h not in result:
            result.append(h)
    return result

# Initialize boundaries
first_positive = next(
    (row[:-1] for row in examples if row[-1].lower() == "yes"),
    None
)

if first_positive is None:
    print("No positive training examples found.")
else:
    S = [first_positive[:]]
    G = [["?" for _ in attributes]]

    for row in examples:
        x, label = row[:-1], row[-1].lower()

        if x == first_positive:
            # First positive example initializes S
            continue

        if label == "yes":
            G = [g for g in G if covers(g, x)]

            new_S = []
            for s in S:
                new_S.append(s if covers(s, x) else generalize(s, x))
            S = remove_duplicates(new_S)

        else:
            S = [s for s in S if not covers(s, x)]

            new_G = []
            for g in G:
                if not covers(g, x):
                    new_G.append(g)
                    continue

                for i in range(len(attributes)):
                    if g[i] == "?":
                        for value in domains[i]:
                            if value != x[i]:
                                h = g[:]
                                h[i] = value
                                if any(more_general(h, s) for s in S):
                                    new_G.append(h)

            G = remove_duplicates(new_G)

    # Display results
    print("Attributes:", attributes)
    print("\nSpecific Boundary (S):")
    for h in S:
        print(h)

    print("\nGeneral Boundary (G):")
    for h in G:
        print(h)

    print("\nCandidate-Elimination completed.")


Attributes: ['Sky', 'AirTemp', 'Humidity', 'Wind', 'Water', 'Forecast']

Specific Boundary (S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

General Boundary (G):
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']

Candidate-Elimination completed.
